# R1.4 — адресные регрессии ядра
Не основной экран автора и не полная статистическая карта. Все числа синтетические. Запуск сверху вниз через nbconvert.

In [ ]:
from pathlib import Path
import os, sys, json
root=Path.cwd()
while root != root.parent and not (root/'efc/model.py').exists():
    root=root.parent
assert (root/'efc/model.py').exists(), 'Сначала получите рабочую ветку репозитория'
os.chdir(root)
sys.path.insert(0,str(root))
from efc.model import Config, simulate, COMPONENTS
print(sys.version)

Один Fog, B=2. Ручные ответы заданы литералами до запуска ядра.

In [ ]:
c=Config(n_edge=1,n_fog=1,n_cloud=1,buffer=2,n_tasks=5,mobile_fraction=0,edge_fixed_ms=(2,2),edge_processing_share=.5,fog_service_ms=(10,10),fog_cloud_ms=(1,1),cloud_service_ms=(1,1))
r=simulate(c,[0,.001,.002,.003,.010])
expected_wait=[0,9,18,None,20]
expected_latency=[14,23,32,None,34]
for i,row in enumerate(r['tasks']):
    if i==3:
        assert row['status']=='dropped' and 'latency_ms' not in row
    else:
        assert abs(row['fog_wait_ms']-expected_wait[i])<1e-9
        assert abs(row['latency_ms']-expected_latency[i])<1e-9
        assert abs(sum(row[k] for k in COMPONENTS)-row['latency_ms'])<1e-9
print('Ручные ответы совпали:',expected_wait,expected_latency)

Модельные интервалы Fog сопоставлены с ручными. Потерянная задача не получает интервал обслуживания.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,3))
manual={0:(2,12),1:(12,22),2:(22,32),4:(32,42)}
for row in r['tasks']:
    i=row['task_id']
    if i in manual:
        a,b=manual[i]
        ax.barh(i,(row['fog_end_s']-row['fog_start_s'])*1000,left=row['fog_start_s']*1000,height=.32,label='Модель' if i==0 else None)
        ax.plot([a,b],[i+.25,i+.25],'--',label='Ручной ответ' if i==0 else None)
    else:
        ax.scatter([row['drop_s']*1000],[i],marker='x')
ax.set_yticks(range(5),[str(i) for i in range(1,6)])
ax.set_xlabel('Время, мс'); ax.set_ylabel('Задача'); ax.legend(); fig.tight_layout()
plt.show()

In [ ]:
out=root/'results/r14/current';out.mkdir(parents=True,exist_ok=True)
(out/'manual-five.json').write_text(json.dumps(r,ensure_ascii=False,indent=2)+'\n')
assert r['metrics']['completed']==4 and r['metrics']['dropped']==1
print('OK: 4 завершения, 1 потеря. Это не проверка статьи/карты. Нейросеть и PDF не запускались.')